In [1]:
pip install requests pandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
import requests
import pandas as pd
import os
import time
import json
from datetime import datetime
from typing import List, Dict, Any, Optional, Tuple

SNAPSHOT_URL = "https://hub.snapshot.org/graphql"

# ==========================
# ✅ CONFIG
# ==========================
SPACES_FILE = "data/processed/selected_spaces_follower_knee.json"

OUT_DIR = "D:\snapshot_votes_441"
SPACE_CSV_DIR = os.path.join(OUT_DIR, "spaces")
PROGRESS_DIR = os.path.join(OUT_DIR, "progress_by_space")
SKIP_DIR = os.path.join(OUT_DIR, "skip_by_space")          # proposal-level skip records
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(SPACE_CSV_DIR, exist_ok=True)
os.makedirs(PROGRESS_DIR, exist_ok=True)
os.makedirs(SKIP_DIR, exist_ok=True)

FAIL_LOG = os.path.join(OUT_DIR, "failures.log")

PROPOSAL_BATCH = 100

# votes page size（起始值，遇到 524/timeout 会自动降载）
VOTES_PAGE_SIZE = 1000
MIN_VOTES_PAGE_SIZE = 50

# 网络/限流参数
SLEEP_BETWEEN_CALLS = 1.6
MAX_RETRIES = 6
BACKOFF_BASE = 2.0
HTTP_TIMEOUT = (10, 200)  # (connect_timeout, read_timeout)

# 小规模测试开关（建议先跑 5 个 space、每个 3 个 proposal）
MAX_SPACES = None
MAX_PROPOSALS_PER_SPACE = None

# 可选：先抓 followersCount 并按重要性排序
SORT_BY_FOLLOWERS = True

# ✅ 方案A：对单个 proposal 最大“失败轮数”（会尝试降载多次，仍失败就跳过）
MAX_PROPOSAL_FAILURE_ROUNDS = 3
PAGE_SIZE_LADDER = [1000, 500, 250, 200, 100, 50]

# ✅ NEW：直接跳过某些“超重/不稳定”spaces
SKIP_SPACES = {"stgdao.eth"}   # 你要求：直接跳过 stgdao
# ==========================


# ---------- Utilities ----------
def safe_filename(name: str) -> str:
    keep = []
    for ch in name:
        if ch.isalnum() or ch in ("-", "_", "."):
            keep.append(ch)
        else:
            keep.append("_")
    return "".join(keep)


def space_csv_path(space: str) -> str:
    return os.path.join(SPACE_CSV_DIR, f"{safe_filename(space)}.csv")


def progress_path_for_space(space: str) -> str:
    return os.path.join(PROGRESS_DIR, f"progress_{safe_filename(space)}.json")


def skip_path_for_space(space: str) -> str:
    return os.path.join(SKIP_DIR, f"skip_{safe_filename(space)}.json")


def log_failure(tag: str, msg: str):
    with open(FAIL_LOG, "a", encoding="utf-8") as f:
        f.write(f"[{tag}] {msg}\n")


def load_json_list(path: str) -> List[Any]:
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)
        if isinstance(obj, list):
            return obj
    return []


def append_skip(space: str, record: Dict[str, Any]):
    path = skip_path_for_space(space)
    arr = load_json_list(path)
    arr.append(record)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(arr, f, ensure_ascii=False, indent=2)


# ---------- Robust GraphQL ----------
def post_gql(query: str, variables: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    payload = {"query": query}
    if variables:
        payload["variables"] = variables

    last_err = None
    last_status = None
    last_text = None

    for attempt in range(MAX_RETRIES):
        try:
            r = requests.post(SNAPSHOT_URL, json=payload, timeout=HTTP_TIMEOUT)
            last_status = r.status_code
            last_text = r.text

            if r.status_code == 400:
                raise RuntimeError(f"HTTP 400 Bad Request: {r.text[:800]}")

            # include 524 as retryable
            if r.status_code in (429, 500, 502, 503, 504, 524):
                raise RuntimeError(f"HTTP {r.status_code}: {r.text[:200]}")

            r.raise_for_status()

            data = r.json()
            if "errors" in data:
                raise RuntimeError(f"GQL errors: {str(data['errors'])[:800]}")

            return data["data"]

        except Exception as e:
            last_err = e

            if isinstance(e, RuntimeError) and "HTTP 400" in str(e):
                print("\n========== SNAPSHOT 400 BAD REQUEST ==========")
                print("Response (first 800 chars):", (last_text or "")[:800])
                print("Query (first 800 chars):", query[:800])
                print("=============================================\n")
                raise

            wait = (BACKOFF_BASE ** attempt) + 0.2 * attempt
            print(f"⚠️ 请求失败（attempt {attempt+1}/{MAX_RETRIES}）：{e} | backoff {wait:.1f}s")
            time.sleep(wait)

    raise RuntimeError(
        f"❌ GraphQL failed after {MAX_RETRIES} retries. "
        f"last_status={last_status}, last_error={repr(last_err)}"
    )


# ---------- Load 441 spaces ----------
def load_selected_space_ids(spaces_file: str) -> List[str]:
    with open(spaces_file, "r", encoding="utf-8") as f:
        obj = json.load(f)

    ids = obj.get("selected_space_ids", [])
    if not isinstance(ids, list):
        raise ValueError("selected_space_ids is not a list in SPACES_FILE")

    seen = set()
    out = []
    for s in ids:
        if isinstance(s, str) and s and s not in seen:
            seen.add(s)
            out.append(s)
    return out


# ---------- Progress per space ----------
def load_progress(space: str) -> set:
    path = progress_path_for_space(space)
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return set(json.load(f))
    return set()


def save_progress(space: str, completed: set):
    path = progress_path_for_space(space)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(sorted(list(completed)), f, ensure_ascii=False, indent=2)


# ---------- Snapshot helpers ----------
def fetch_space_followers(space: str) -> Optional[int]:
    q = f"""
    {{
      space(id: "{space}") {{
        followersCount
      }}
    }}
    """
    data = post_gql(q)
    sp = data.get("space")
    if not sp:
        return None
    fc = sp.get("followersCount")
    try:
        return int(fc) if fc is not None else None
    except:
        return None


def fetch_all_proposals(space: str) -> List[Dict[str, Any]]:
    all_props = []
    skip = 0

    print(f"\n🚀 开始抓取 Space={space} 的全部 proposals ...")
    while True:
        q = f"""
        {{
          proposals(first: {PROPOSAL_BATCH}, skip: {skip},
            where: {{ space_in: ["{space}"] }},
            orderBy: "created", orderDirection: desc
          ) {{
            id
            title
            body
            choices
            author
            state
            created
          }}
        }}
        """
        data = post_gql(q)
        batch = data.get("proposals", [])
        print(f"📦 {space} proposals: {skip + 1} ~ {skip + len(batch)}")

        if not batch:
            break

        all_props.extend(batch)
        if len(batch) < PROPOSAL_BATCH:
            break

        skip += PROPOSAL_BATCH
        time.sleep(SLEEP_BETWEEN_CALLS)

    print(f"✅ {space} proposals 总数：{len(all_props)}")
    return all_props


def _is_retryable_heavy_error(e: Exception) -> bool:
    msg = str(e).lower()
    keywords = [
        "524", "504", "502", "503", "500", "server error", "gateway",
        "read timed out", "timeout", "timed out", "connectionpool", "connection aborted"
    ]
    return any(k in msg for k in keywords)


def fetch_all_votes_for_proposal(proposal_id: str, space: str) -> Tuple[List[Dict[str, Any]], bool]:
    """
    Cursor pagination via created_lt (desc) to bypass skip<=5000 limit.
    ✅ Dynamic page_size reduction + multi-round attempts.
    If still failing after several rounds, SKIP this proposal and continue.
    Returns: (votes, skipped_flag)
    """
    page_size = VOTES_PAGE_SIZE

    for round_idx in range(1, MAX_PROPOSAL_FAILURE_ROUNDS + 1):
        all_votes: List[Dict[str, Any]] = []
        created_lt = int(time.time()) + 10
        seen = set()

        print(f"   ↪️ votes fetch round {round_idx}/{MAX_PROPOSAL_FAILURE_ROUNDS} | page_size={page_size}")

        try:
            while True:
                q = f"""
                {{
                  votes(first: {page_size},
                    where: {{ proposal: "{proposal_id}", created_lt: {created_lt} }},
                    orderBy: "created", orderDirection: desc
                  ) {{
                    voter
                    choice
                    vp
                    created
                  }}
                }}
                """
                data = post_gql(q)
                batch = data.get("votes", [])

                if not batch:
                    break

                for v in batch:
                    key = (v.get("voter"), v.get("created"))
                    if key in seen:
                        continue
                    seen.add(key)
                    all_votes.append(v)

                last_created = batch[-1]["created"]
                if last_created >= created_lt:
                    break
                created_lt = last_created

                if len(batch) < page_size:
                    break

                time.sleep(SLEEP_BETWEEN_CALLS)

            return all_votes, False

        except Exception as e:
            if _is_retryable_heavy_error(e):
                smaller = [x for x in PAGE_SIZE_LADDER if x < page_size]
                if smaller:
                    page_size = max(MIN_VOTES_PAGE_SIZE, smaller[0])
                else:
                    page_size = max(MIN_VOTES_PAGE_SIZE, page_size // 2)

                print(f"⚠️ Proposal 重型/超时：{e}")
                print(f"   ✅ 降载：votes page_size → {page_size}，稍等后继续该 proposal ...")
                time.sleep(6.0 + 2.0 * round_idx)
                continue

            raise

    append_skip(space, {
        "proposal_id": proposal_id,
        "reason": f"Failed after {MAX_PROPOSAL_FAILURE_ROUNDS} rounds with page_size reductions (likely 524/timeout heavy proposal)"
    })
    print(f"🚫 已跳过该 proposal（已记录 skip）：{proposal_id}")
    return [], True


def compute_majority_choice(votes: List[Dict[str, Any]]) -> Optional[int]:
    counter: Dict[int, int] = {}
    for v in votes:
        c = v.get("choice")
        if isinstance(c, int):
            counter[c] = counter.get(c, 0) + 1
        elif isinstance(c, dict):
            for k, val in c.items():
                try:
                    kk = int(k)
                except:
                    continue
                if val and val > 0:
                    counter[kk] = counter.get(kk, 0) + 1
    if not counter:
        return None
    return max(counter.items(), key=lambda x: x[1])[0]


def append_rows_to_space_csv(space: str, rows: List[Dict[str, Any]]):
    if not rows:
        return
    path = space_csv_path(space)
    df = pd.DataFrame(rows)
    header = not os.path.exists(path)
    df.to_csv(path, mode="a", index=False, header=header)


# ---------- MAIN ----------
def main():
    spaces = load_selected_space_ids(SPACES_FILE)
    print(f"✅ 从 selected_space_ids 读取 spaces 数量：{len(spaces)}")

    # remove skipped spaces early (optional), still keep SKIP_SPACES in loop for safety
    spaces = [s for s in spaces if s not in SKIP_SPACES]
    print(f"⏭️  已从列表移除 SKIP_SPACES={SKIP_SPACES} | remaining={len(spaces)}")

    if MAX_SPACES is not None:
        spaces = spaces[:MAX_SPACES]
        print(f"🧪 MAX_SPACES={MAX_SPACES} → 本次只跑前 {len(spaces)} 个 spaces")

    space_items: List[Tuple[str, Optional[int]]] = [(s, None) for s in spaces]
    if SORT_BY_FOLLOWERS:
        print("🔎 获取 followersCount 并排序（None 放最后）...")
        tmp = []
        for i, s in enumerate(spaces, 1):
            try:
                fc = fetch_space_followers(s)
            except Exception as e:
                fc = None
                log_failure("FOLLOWERS_FAIL", f"{s} | {repr(e)}")
            tmp.append((s, fc))
            if i % 25 == 0:
                print(f"   ... followers fetched {i}/{len(spaces)}")
            time.sleep(0.2)
        tmp.sort(key=lambda x: (x[1] is None, -(x[1] or 0)))
        space_items = tmp
        print("✅ 排序后前 5 个：", space_items[:5])

    for idx, (space, followers) in enumerate(space_items, 1):
        # extra safety
        if space in SKIP_SPACES:
            log_failure("SPACE_SKIPPED", f"{space} | skipped by user request")
            print(f"⏭️  已跳过 space：{space}（原因：手动跳过）")
            continue

        print("\n" + "=" * 90)
        print(f"🏷️  Space [{idx}/{len(space_items)}]: {space} | followersCount={followers}")
        print("=" * 90)

        completed = load_progress(space)
        print(f"📌 已完成 proposals 数：{len(completed)}（{progress_path_for_space(space)}）")
        print(f"📄 输出 CSV：{space_csv_path(space)}")
        print(f"🧾 skip 记录：{skip_path_for_space(space)}")

        try:
            proposals = fetch_all_proposals(space)
        except Exception as e:
            log_failure("SPACE_FAIL", f"{space} | {repr(e)}")
            print(f"❌ Space 抓 proposals 失败（已记录 failures.log），跳过：{e}")
            continue

        if MAX_PROPOSALS_PER_SPACE is not None:
            proposals = proposals[:MAX_PROPOSALS_PER_SPACE]
            print(f"🧪 MAX_PROPOSALS_PER_SPACE={MAX_PROPOSALS_PER_SPACE} → 本次只跑 {len(proposals)} 个 proposals")

        for i, p in enumerate(proposals, 1):
            pid = p["id"]
            if pid in completed:
                continue

            title = (p.get("title") or "")[:80]
            print(f"\n🔍 [{space}] Proposal {i}/{len(proposals)}：{title}")

            try:
                votes, skipped = fetch_all_votes_for_proposal(pid, space=space)
                if skipped:
                    completed.add(pid)
                    save_progress(space, completed)
                    continue

                print(f"🧾 votes 数：{len(votes)}")

                if not votes:
                    completed.add(pid)
                    save_progress(space, completed)
                    continue

                total_vp = sum(v.get("vp", 0) for v in votes)
                if total_vp == 0:
                    completed.add(pid)
                    save_progress(space, completed)
                    continue

                majority = compute_majority_choice(votes)
                if majority is None:
                    completed.add(pid)
                    save_progress(space, completed)
                    continue

                rows = []
                for v in votes:
                    vp = v.get("vp", 0)
                    vp_ratio = vp / total_vp if total_vp else 0

                    choice_used = None
                    aligned = False
                    c = v.get("choice")

                    if isinstance(c, int):
                        choice_used = c
                        aligned = (c == majority)
                    elif isinstance(c, dict):
                        choice_used = json.dumps(c, ensure_ascii=False)
                        aligned = (str(majority) in c and c[str(majority)] > 0)

                    rows.append({
                        "Space": space,
                        "FollowersCount": followers,
                        "Proposal ID": pid,
                        "Proposal Title": p.get("title", ""),
                        "Proposal Body": (p.get("body") or "").replace("\n", " ").replace("\r", " ")[:1000],
                        "Created Time": datetime.utcfromtimestamp(p["created"]).isoformat(),
                        "Voter": v.get("voter"),
                        "Choice": choice_used,
                        "Voting Power": vp,
                        "VP Ratio (%)": round(vp_ratio * 100, 4),
                        "Is Whale": vp_ratio > 0.05,
                        "Aligned With Majority": aligned,
                        "Vote Timestamp": datetime.utcfromtimestamp(v["created"]).isoformat()
                    })

                append_rows_to_space_csv(space, rows)

                completed.add(pid)
                save_progress(space, completed)

                print(f"✅ 写入 {len(rows)} 行 | space 完成 proposals={len(completed)}")
                time.sleep(SLEEP_BETWEEN_CALLS)

            except Exception as e:
                log_failure("PROPOSAL_FAIL", f"{space} | {pid} | {repr(e)}")
                print(f"❌ Proposal 失败（已记录 failures.log），继续：{e}")
                continue

    print("\n🎉 全部完成（或已尽可能完成）！")
    print(f"📁 Space CSV 目录：{SPACE_CSV_DIR}")
    print(f"📌 progress_dir：{PROGRESS_DIR}")
    print(f"🧾 skip_dir：{SKIP_DIR}")
    print(f"📄 failures：{FAIL_LOG}")
    print(f"⏭️  skipped spaces：{sorted(list(SKIP_SPACES))}")


if __name__ == "__main__":
    main()


✅ 从 selected_space_ids 读取 spaces 数量：441
⏭️  已从列表移除 SKIP_SPACES={'stgdao.eth'} | remaining=440
🔎 获取 followersCount 并排序（None 放最后）...
   ... followers fetched 25/440
   ... followers fetched 50/440
   ... followers fetched 75/440
   ... followers fetched 100/440
⚠️ 请求失败（attempt 1/6）：HTTP 429: {"error":"unauthorized","error_description":"too many requests, refer to https://docs.snapshot.box/tools/api/api-keys#limits"} | backoff 1.0s
⚠️ 请求失败（attempt 2/6）：HTTP 429: {"error":"unauthorized","error_description":"too many requests, refer to https://docs.snapshot.box/tools/api/api-keys#limits"} | backoff 2.2s
⚠️ 请求失败（attempt 3/6）：HTTP 429: {"error":"unauthorized","error_description":"too many requests, refer to https://docs.snapshot.box/tools/api/api-keys#limits"} | backoff 4.4s
⚠️ 请求失败（attempt 4/6）：HTTP 429: {"error":"unauthorized","error_description":"too many requests, refer to https://docs.snapshot.box/tools/api/api-keys#limits"} | backoff 8.6s
⚠️ 请求失败（attempt 5/6）：HTTP 429: {"error":"unautho


🔍 [arbitrumfoundation.eth] Proposal 17/399：[Constitutional] AIP: Disable Legacy Tether Bridge
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：3265
✅ 写入 3265 行 | space 完成 proposals=17

🔍 [arbitrumfoundation.eth] Proposal 18/399：Updating the Code of Conduct & DAO’s Procedures
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：2867
✅ 写入 2867 行 | space 完成 proposals=18

🔍 [arbitrumfoundation.eth] Proposal 19/399：[CONSTITUTIONAL] Register $BORING in the Arbitrum generic-custom gateway
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：2996
✅ 写入 2996 行 | space 完成 proposals=19

🔍 [arbitrumfoundation.eth] Proposal 20/399：[Constitutional] AIP: Update the Upgrade Executors
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：2986
✅ 写入 2986 行 | space 完成 proposals=20

🔍 [arbitrumfoundation.eth] Proposal 21/399：Entropy Advisors: Exclusively Working with the Arbitrum DAO, Year 2 and Year 3
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：2946
✅ 写入 2946 行 | space 完成 proposals=2

🧾 votes 数：4194
✅ 写入 4194 行 | space 完成 proposals=58

🔍 [arbitrumfoundation.eth] Proposal 59/399：ARDC (V2) Security Election
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：4271
✅ 写入 4271 行 | space 完成 proposals=59

🔍 [arbitrumfoundation.eth] Proposal 60/399：ARDC (V2) Risk Election
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：4168
✅ 写入 4168 行 | space 完成 proposals=60

🔍 [arbitrumfoundation.eth] Proposal 61/399：ARDC (V2) Research Election
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：4099
✅ 写入 4099 行 | space 完成 proposals=61

🔍 [arbitrumfoundation.eth] Proposal 62/399：ARDC (V2) Supervisory Council Election
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：3989
✅ 写入 3989 行 | space 完成 proposals=62

🔍 [arbitrumfoundation.eth] Proposal 63/399：[NON-CONSTITUTIONAL] Arbitrum Onboarding V2: A Governance Bootcamp
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：4159
✅ 写入 4159 行 | space 完成 proposals=63

🔍 [arbitrumfoundation.eth] Proposal 64/399：Arbitrum D.A.O. Dom

🧾 votes 数：4042
✅ 写入 4042 行 | space 完成 proposals=102

🔍 [arbitrumfoundation.eth] Proposal 103/399：Transparency and Standardized Metrics for Orbit Chains
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：4022
✅ 写入 4022 行 | space 完成 proposals=103

🔍 [arbitrumfoundation.eth] Proposal 104/399：ARB Staking: Unlock ARB Utility and Align Governance 
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：25119
✅ 写入 25119 行 | space 完成 proposals=104

🔍 [arbitrumfoundation.eth] Proposal 105/399：[Non-constitutional] Incentives Detox Proposal
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：3687
✅ 写入 3687 行 | space 完成 proposals=105

🔍 [arbitrumfoundation.eth] Proposal 106/399： Change Arbitrum Expansion Program to allow deployments of new Orbit chains on a
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：4088
✅ 写入 4088 行 | space 完成 proposals=106

🔍 [arbitrumfoundation.eth] Proposal 107/399：Furucombo's Misuse of Funds
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：4296
✅ 写入 429

🧾 votes 数：6296
✅ 写入 6296 行 | space 完成 proposals=146

🔍 [arbitrumfoundation.eth] Proposal 147/399：Pilot Phase: M&A for Arbitrum DAO
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：7529
✅ 写入 7529 行 | space 完成 proposals=147

🔍 [arbitrumfoundation.eth] Proposal 148/399：Proposal for Approval of DeDaub as the ADPC Security Advisor
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：123531
✅ 写入 123531 行 | space 完成 proposals=148

🔍 [arbitrumfoundation.eth] Proposal 149/399： Buffer - LTIPP [Post Council Feedback]
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：31070
✅ 写入 31070 行 | space 完成 proposals=149

🔍 [arbitrumfoundation.eth] Proposal 150/399：Sushi - LTIPP [Post Council Feedback]
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：18348
✅ 写入 18348 行 | space 完成 proposals=150

🔍 [arbitrumfoundation.eth] Proposal 151/399：Grant Request - Curve Finance
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：28775
✅ 写入 28775 行 | space 完成 proposals=151

🔍 [arbitrumfoundation.et


🔍 [arbitrumfoundation.eth] Proposal 190/399：Knights of the Ether LTIPP Council Recommended Proposal
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：5665
✅ 写入 5665 行 | space 完成 proposals=190

🔍 [arbitrumfoundation.eth] Proposal 191/399：KelpDAO LTIPP Council Recommended Proposal
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：5852
✅ 写入 5852 行 | space 完成 proposals=191

🔍 [arbitrumfoundation.eth] Proposal 192/399：IPOR Protocol LTIPP Council Recommended Proposal
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：5617
✅ 写入 5617 行 | space 完成 proposals=192

🔍 [arbitrumfoundation.eth] Proposal 193/399：Integral LTIPP Council Recommended Proposal
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：5541
✅ 写入 5541 行 | space 完成 proposals=193

🔍 [arbitrumfoundation.eth] Proposal 194/399：Index Coop LTIPP Council Recommended Proposal
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：5500
✅ 写入 5500 行 | space 完成 proposals=194

🔍 [arbitrumfoundation.eth] Proposal 195/399：Hop LTI

🧾 votes 数：5132
✅ 写入 5132 行 | space 完成 proposals=233

🔍 [arbitrumfoundation.eth] Proposal 234/399：Steer LTIPP Council Recommended Proposal
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：5177
✅ 写入 5177 行 | space 完成 proposals=234

🔍 [arbitrumfoundation.eth] Proposal 235/399：STFX LTIPP Council Recommended Proposal
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：5143
✅ 写入 5143 行 | space 完成 proposals=235

🔍 [arbitrumfoundation.eth] Proposal 236/399：SX Bet LTIPP Council Recommended Proposal
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：5122
✅ 写入 5122 行 | space 完成 proposals=236

🔍 [arbitrumfoundation.eth] Proposal 237/399：Symbiosis LTIPP Council Recommended Proposal
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：5287
✅ 写入 5287 行 | space 完成 proposals=237

🔍 [arbitrumfoundation.eth] Proposal 238/399：Synonym Finance LTIPP Council Recommended Proposal
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：5273
✅ 写入 5273 行 | space 完成 proposals=238

🔍 [arbitrumfoundat

✅ 写入 50034 行 | space 完成 proposals=276

🔍 [arbitrumfoundation.eth] Proposal 277/399：Non-Constitutional AIP: Arbitrum Security Enhancement Fund
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：50718
✅ 写入 50718 行 | space 完成 proposals=277

🔍 [arbitrumfoundation.eth] Proposal 278/399：Proposal: Activate ARB Staking
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：40512

🔍 [arbitrumfoundation.eth] Proposal 279/399：Proposal: Build Optimal Onboarding for STIP Teams (BOOST)
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：43564
✅ 写入 43564 行 | space 完成 proposals=279

🔍 [arbitrumfoundation.eth] Proposal 280/399：Proposal: Empowering Early Contributors: The community Arbiter Proposal
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：44524
✅ 写入 44524 行 | space 完成 proposals=280

🔍 [arbitrumfoundation.eth] Proposal 281/399：UniDex STIP Proposal - Round 1
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：32494
✅ 写入 32494 行 | space 完成 proposals=281

🔍 [arbitrumfoundation.eth] P


🔍 [arbitrumfoundation.eth] Proposal 322/399：Solv Protocol STIP Proposal - Round 1
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：17504
✅ 写入 17504 行 | space 完成 proposals=322

🔍 [arbitrumfoundation.eth] Proposal 323/399：REALM STIP Proposal - Round 1
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：17138
✅ 写入 17138 行 | space 完成 proposals=323

🔍 [arbitrumfoundation.eth] Proposal 324/399：Wombex Finance STIP Proposal - Round 1
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：17424
✅ 写入 17424 行 | space 完成 proposals=324

🔍 [arbitrumfoundation.eth] Proposal 325/399：dForce STIP Proposal - Round 1
   ↪️ votes fetch round 1/3 | page_size=1000
🧾 votes 数：17271


OSError: [Errno 28] No space left on device: 'data/processed/votes_441_per_space\\failures.log'